In [ ]:
import os
import torch
import matplotlib.pyplot as plt
import numpy as np

# Set plot style
plt.style.use("seaborn-v0_8-whitegrid")

# Increase font sizes for better visibility
plt.rcParams.update({
    'font.size': 14,
    'axes.titlesize': 18,
    'axes.labelsize': 16,
    'xtick.labelsize': 14,
    'ytick.labelsize': 14,
    'legend.fontsize': 12,
    'figure.titlesize': 20
})

In [ ]:
class ResultLoader:
    def __init__(self, base_dir="results"):
        self.base_dir = base_dir
        self.algo_patterns = {
            "fedala": lambda args: f"_{args['eta']}_{args['rand_percent']}_{args['layer_idx']}_{args['ala_threshold']}_{args['num_pre_loss']}",
            "fedavg": lambda args: "",
            "feddyn": lambda args: f"_alpha{args['alpha_coef']}",
            "feddpl": lambda args: f"_{args['lambda_']}_{args['epoch_pln']}_{args['lr_pln']}_{args['batch_size_pln']}_{args['depth_pln']}_{args['width_pln']}_{args['mode']}_{args['fixed_proto']}_{args['init_emb']}_{args['har']}",
            "feddpl1": lambda args: f"_{args['lambda_']}_{args['epoch_pln']}_{args['lr_pln']}_{args['batch_size_pln']}_{args['depth_pln']}_{args['width_pln']}_{args['mode']}_{args['fixed_proto']}_{args['init_emb']}_{args['har']}",
            "fedfm": lambda args: f"_{args['mu']}",
            "fedkd": lambda args: f"_{args['lr_g']}_{args['energy']}",
            "fedlsa": lambda args: f"_{args['lambda_com']}_{args['alpha_sep']}_{args['server_epochs']}_{args['server_lr']}_{args['tau']}",
            "fedper": lambda args: "",
            "fedpln": lambda args: f"_{args['lambda_']}_{args['epoch_pln']}_{args['lr_pln']}_{args['batch_size_pln']}_{args['depth_pln']}_{args['width_pln']}_{args['mode']}_{args['fixed_proto']}_{args['init_emb']}_{args['har']}",
            "fedproc": lambda args: "",
            "fedproto": lambda args: f"_{args['mu']}",
            "fedprox": lambda args: f"_{args['mu']}",
            "fedrep": lambda args: f"_{args['epochs_head']}",
            "fedsa": lambda args: f"_{args['alpha_sa']}_{args['lambda_r']}_{args['lambda_mcl']}_{args['lambda_cc']}",
            "scaffold": lambda args: f"_glr{args['global_lr']}",
            "fedtgp": lambda args: f"_{args['lamda_']}_{args['server_epochs']}_{args['server_lr']}_{args['margin_threshold']}",
            "fml": lambda args: f"_{args['alpha_fml']}_{args['beta_fml']}",
            "lgfedavg": lambda args: "",
            "moon": lambda args: f"_{args['mu']}_{args['tau']}",
            "proxyfl": lambda args: f"_{args['mu']}_{args['adj_type']}",
            "fedtest": lambda args: f"_{args['mu_test']}",
            "local": lambda args: "_local",
        }

    def _average_recursive(self, data_list):
        if not data_list: return None
        first = data_list[0]
        if isinstance(first, list):
            min_len = min(len(d) for d in data_list)
            return np.mean(np.array([d[:min_len] for d in data_list]), axis=0).tolist()
        elif isinstance(first, dict):
            res = {}
            for key in first.keys():
                sub_list = [d[key] for d in data_list if isinstance(d, dict) and key in d]
                if sub_list: res[key] = self._average_recursive(sub_list)
            return res
        return first

    def load(self, algo, dataset, partition, num_clients, specific_run=None, **kwargs):
        folder_name = f"{dataset}_{partition}_{num_clients}"
        if partition == "dirichlet": folder_name += f"_{kwargs.get('alpha', 0.1)}"
        elif partition == "pathological": folder_name += f"_{kwargs.get('n_class', 2)}"

        folder_path = os.path.join(self.base_dir, algo, folder_name)
        if not os.path.exists(folder_path):
            print(f"  [Warning] Folder not found: {folder_path}")
            return None

        base_name = f"{kwargs.get('epochs', 10)}_{kwargs.get('batch_size', 64)}_{kwargs.get('lr', 0.01)}"
        suffix_gen = self.algo_patterns.get(algo)
        if suffix_gen:
            try: base_name += suffix_gen(kwargs)
            except KeyError as e:
                print(f"  [Error] Missing parameter {e} for algo {algo}")
                return None

        if specific_run is not None: run_indices = [specific_run]
        else:
            run_indices = []
            idx = 0
            while os.path.exists(os.path.join(folder_path, f"{base_name}_{idx}.pt")):
                run_indices.append(idx)
                idx += 1

        if not run_indices:
            print(f"  [Warning] No files found for: {base_name}_*.pt in {folder_path}")
            return None

        loaded_data = []
        for idx in run_indices:
            file_path = os.path.join(folder_path, f"{base_name}_{idx}.pt")
            try:
                loaded_data.append(torch.load(file_path, map_location="cpu"))
            except Exception as e:
                print(f"  [Error] Failed to load {file_path}: {e}")

        if not loaded_data: return None
        if len(loaded_data) == 1: return loaded_data[0]

        avg_result = {}
        for key in ["acc", "loss"]:
            if key in loaded_data[0]:
                avg_result[key] = self._average_recursive([d[key] for d in loaded_data if key in d])
        return avg_result

In [ ]:
def plot_results(results_dict, metric="acc", title=None, xlabel="Rounds", ylabel="Accuracy", max_rounds=None):
    plt.figure(figsize=(12, 7))
    for label, data in results_dict.items():
        if data is None: continue
        metric_data = data.get(metric)
        if metric_data is None: continue
        if isinstance(metric_data, list):
            y = metric_data[:max_rounds] if max_rounds else metric_data
            plt.plot(y, label=f"{label} (Max: {max(y):.2f})")
        elif isinstance(metric_data, dict):
            if "model" in metric_data:
                y = metric_data["model"][:max_rounds] if max_rounds else metric_data["model"]
                plt.plot(y, label=f"{label}-Model (Max: {max(y):.2f})")
            if "proto" in metric_data:
                y = metric_data["proto"][:max_rounds] if max_rounds else metric_data["proto"]
                plt.plot(y, linestyle="--", alpha=0.8, label=f"{label}-Proto (Max: {max(y):.2f})")

    plt.title(title or f"Comparison of {metric.upper()}")
    plt.xlabel(xlabel)
    plt.ylabel(ylabel)
    plt.legend()
    plt.grid(True, alpha=0.3)
    plt.tight_layout()

    # --- Save Logic ---
    if not os.path.exists("figures"): os.makedirs("figures")
    save_name = (title or "comparison").lower().replace(" ", "_").replace("(", "").replace(")", "") + ".png"
    plt.savefig(os.path.join("figures", save_name), dpi=300)
    print(f"Figure saved to figures/{save_name}")
    plt.show()

In [ ]:
loader = ResultLoader("../results")
common_args = {"dataset": "cifar10", "partition": "dirichlet", "num_clients": 10, "epochs": 10, "batch_size": 64, "lr": 0.01, "alpha": 0.1, "n_class": 2}

experiments = {
    "FedAvg": ("fedavg", {}),
    "FedProx": ("fedprox", {"mu": 0.01}),
    "Scaffold": ("scaffold", {"global_lr": 1.0}),
    "FedDyn": ("feddyn", {"alpha_coef": 0.1}),
    "FedPLN": ("fedpln", {"lambda_": 10.0, "epoch_pln": 10, "lr_pln": 0.01, "batch_size_pln": 64, "depth_pln": 1, "width_pln": 512, "mode": "normal", "har": 0, "fixed_proto": 0, "init_emb": 0}),
    "FedProc": ("fedproc", {}),
    "FedFM": ("fedfm", {"mu": 1.0}),
    "MOON": ("moon", {"mu": 0.01, "tau": 0.5}),
    "FedSA": ("fedsa", {"alpha_sa": 0.9999, "lambda_r": 0.1, "lambda_mcl": 0.01, "lambda_cc": 1.0}),
    "FedLSA": ("fedlsa", {"lambda_com": 0.1, "alpha_sep": 0.1, "server_epochs": 10, "server_lr": 0.01, "tau": 0.1}),

    "FedPer": ("fedper", {}),
    "FedRep": ("fedrep", {"epochs_head": 5}),
    "FedProto": ("fedproto", {"mu": 1.0}),
    "FedALA": ("fedala", {"eta": 1.0, "rand_percent": 80, "layer_idx": 2, "ala_threshold": 0.1, "num_pre_loss": 10}),
    "FedTGP": ("fedtgp", {"lamda_": 10.0, "server_epochs": 10, "server_lr": 0.01, "margin_threshold": 1.0}),
    "FedDPL": ("feddpl", {"lambda_": 10.0, "epoch_pln": 10, "lr_pln": 0.01, "batch_size_pln": 64, "depth_pln": 1, "width_pln": 512, "mode": "normal", "fixed_proto": 0, "init_emb": 0, "har": 0}),
    "FedDPL1": ("feddpl1", {"lambda_": 10.0, "epoch_pln": 10, "lr_pln": 0.01, "batch_size_pln": 64, "depth_pln": 1, "width_pln": 512, "mode": "normal", "fixed_proto": 0, "init_emb": 0, "har": 0}),
    "LG-FedAvg": ("lgfedavg", {}),
    "FedKD": ("fedkd", {"lr_g": 0.01, "energy": 0.9}),
    "FML": ("fml", {"alpha_fml": 1.0, "beta_fml": 1.0}),
    "ProxyFL": ("proxyfl", {"mu": 1.0, "adj_type": "ring"}),
    "FedTest": ("fedtest", {"mu_test": 0.1}),
    "Local": ("local", {}),
}

In [ ]:
base_algos = ["FedAvg", "FedProx", "Scaffold", "FedDyn", "FedPLN", "FedProc", "FedFM", "MOON", "FedLSA"]
personalized_algos = ["FedPer", "FedRep", "FedProto", "FedALA", "FedTGP", "FedDPL", "FedDPL1", "LG-FedAvg", "FedKD", "FML", "ProxyFL", "FedSA", "FedTest", "Local"]
selected_group = personalized_algos # Change this to compare different groups

results = {}
for label in selected_group:
    if label not in experiments: continue
    algo_name, kwargs = experiments[label]
    data = loader.load(algo_name, **{**common_args, **kwargs}, specific_run=0)
    if data: results[label] = data

if results:
    plot_results(results, metric="acc", title=f"Test Accuracy on {common_args['dataset']} ({common_args['partition']})")
else:
    print("\n[Error] No results loaded. Check the warnings above for path/parameter mismatches.")